# Optuna

Проверяем те же модели, что в `classical_models.ipynb`, но подбираем гиперпараметры через Optuna вместо `GridSearchCV` и `RandomizedSearchCV`. Optuna смотрит на результаты уже пройденных попыток и предлагает следующее сочетание параметров в сторону более удачной области, а не по сетке и не наугад, поэтому часто находит не худший результат за меньшее число попыток

Признаки, предобработка и схема валидации те же, что в прошлом ноутбуке, принятая предобработка подключается из модуля `src/preprocessing.py`

## 1. Признаки из прошлых ноутбуков

Все шаги, принятые в `baseline_and_preprocessing.ipynb` и `feature_engineering.ipynb`

In [1]:
import sys
from collections.abc import Callable
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import Pipeline, make_pipeline

optuna.logging.set_verbosity(optuna.logging.WARNING)
sys.path.append("..")

from src.preprocessing import OUTLIER_IDS, apply_accepted_preprocessing, build_preprocessor
from src.validation import rmse

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("../data")
TARGET = "SalePrice"
RANDOM_STATE = 42

train = pd.read_csv(DATA_DIR / "train.csv")
X = train.drop(columns=[TARGET, "Id"])
y = train[TARGET]

In [2]:
X_base = apply_accepted_preprocessing(X, DATA_DIR / "train.csv")

keep = ~train["Id"].isin(OUTLIER_IDS)
X_base, y_base = X_base[keep].reset_index(drop=True), y[keep].reset_index(drop=True)

print(f"признаков: {X_base.shape[1]}, домов: {X_base.shape[0]}")

признаков: 69, домов: 1458


## 2. Схема валидации

Та же схема, что в прошлых ноутбуках: 5 фолдов с 3 повторами на фиксированном разбиении, RMSE на `log1p` цены

In [3]:
def cross_validate(model: Pipeline, X: pd.DataFrame, y: pd.Series) -> tuple[float, float]:
    """Считает RMSE на логарифме цены по повторной кросс-валидации

    Возвращает среднее и стандартное отклонение RMSE по всем фолдам
    """
    folds = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
    log_y = np.log1p(y)
    scores = []
    for fit_idx, valid_idx in folds.split(X):
        fitted = clone(model).fit(X.iloc[fit_idx], log_y.iloc[fit_idx])
        prediction = fitted.predict(X.iloc[valid_idx])
        scores.append(rmse(log_y.iloc[valid_idx], prediction))
    return float(np.mean(scores)), float(np.std(scores))


results = []


def optuna_tune_and_evaluate(
    experiment: str,
    build_model: Callable[[optuna.Trial], Pipeline],
    n_trials: int,
    n_startup_trials: int = 10,
    X: pd.DataFrame = X_base,
    y: pd.Series = y_base,
) -> optuna.Study:
    """Подбирает гиперпараметры через Optuna и записывает лучший результат в results

    build_model получает optuna.Trial и должен вернуть готовый пайплайн для этой
    попытки, вызывая trial.suggest_* внутри себя. Ищем минимум RMSE на логарифме
    цены по той же кросс-валидации, что и везде в этом ноутбуке. n_startup_trials
    это число полностью случайных попыток до того, как TPESampler начинает
    опираться на найденные закономерности, для параметров с категориями вроде
    criterion его стоит увеличивать, иначе ранняя случайная удача одной категории
    может исказить весь дальнейший поиск, как случилось с деревом решений
    """

    def objective(trial: optuna.Trial) -> float:
        model = build_model(trial)
        rmse_value, _ = cross_validate(model, X, y)
        return rmse_value

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=n_startup_trials)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials)

    best_model = build_model(optuna.trial.FixedTrial(study.best_params))
    _, std_value = cross_validate(best_model, X, y)
    results.append({"experiment": experiment, "rmse": study.best_value, "std": std_value})
    print(f"{experiment}: RMSE {study.best_value:.4f}, std {std_value:.4f}, лучшие параметры {study.best_params}")
    return study


def show_results() -> pd.DataFrame:
    """Собирает результаты всех экспериментов в таблицу, в порядке добавления"""
    results_df = pd.DataFrame(results)
    experiment_order = results_df["experiment"].drop_duplicates()
    return results_df.set_index("experiment").loc[experiment_order, ["rmse", "std"]].round(4)

## 3. Линейные модели

Обычная линейная регрессия без регуляризации гиперпараметров не имеет смысл, искать нечего, её результат из прошлого ноутбука, RMSE 0.1198, остаётся точкой сравнения

Для Ridge, Lasso и ElasticNet Optuna сама выбирает `alpha` внутри логарифмической шкалы, а для ElasticNet ещё и `l1_ratio`, в тех же границах, что и в `RandomizedSearchCV` прошлого ноутбука

In [4]:
def build_ridge(trial: optuna.Trial) -> Pipeline:
    """Собирает пайплайн Ridge с alpha, подобранным Optuna в логарифмической шкале"""
    alpha = trial.suggest_float("alpha", 1e-2, 1e3, log=True)
    return make_pipeline(build_preprocessor(scale=True), Ridge(alpha=alpha))


ridge_study = optuna_tune_and_evaluate("1. Ridge", build_ridge, n_trials=30)

1. Ridge: RMSE 0.1117, std 0.0060, лучшие параметры {'alpha': 12.521885136283695}


In [5]:
def build_lasso(trial: optuna.Trial) -> Pipeline:
    """Собирает пайплайн Lasso с alpha, подобранным Optuna в логарифмической шкале"""
    alpha = trial.suggest_float("alpha", 1e-4, 10, log=True)
    return make_pipeline(build_preprocessor(scale=True), Lasso(alpha=alpha, max_iter=20000))


lasso_study = optuna_tune_and_evaluate("2. Lasso", build_lasso, n_trials=30)

2. Lasso: RMSE 0.1110, std 0.0052, лучшие параметры {'alpha': 0.0005078929047587406}


In [6]:
def build_elasticnet(trial: optuna.Trial) -> Pipeline:
    """Собирает пайплайн ElasticNet с alpha и l1_ratio, подобранными Optuna"""
    alpha = trial.suggest_float("alpha", 1e-4, 10, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)
    return make_pipeline(build_preprocessor(scale=True), ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000))


elasticnet_study = optuna_tune_and_evaluate("3. ElasticNet", build_elasticnet, n_trials=40)

3. ElasticNet: RMSE 0.1111, std 0.0053, лучшие параметры {'alpha': 0.0009008366097856419, 'l1_ratio': 0.5103587059088599}


### Выводы: линейные модели

- Ridge: RMSE 0.1117, alpha 12.52. В `classical_models.ipynb` `GridSearchCV` на 30 точках сетки нашёл alpha 12.69 и тот же RMSE 0.1117. Optuna на том же числе попыток, 30, пришла к практически той же точке
- Lasso: RMSE 0.1110, чуть лучше, чем 0.1111 у `RandomizedSearchCV` в прошлом ноутбуке, alpha тоже почти совпадает, 0.000508 против 0.000489
- ElasticNet: RMSE 0.1111, совпадает с прошлым ноутбуком, но `l1_ratio` найден совсем другой, 0.51 против 0.95 у `RandomizedSearchCV`. Два разных сочетания alpha и l1_ratio дают одинаковое качество
- На линейных моделях с одним-двумя гиперпараметрами разница между случайным перебором по сетке и Optuna почти не видна: пространство маленькое, и оба способа его покрывают. Разница должна стать заметнее на моделях с гиперпараметрами из документации вроде деревьев и бустингов, там пространство намного больше

## 4. KNN

Те же три содержательных гиперпараметра, что и в прошлом ноутбуке: число соседей, способ взвешивания и степень расстояния Минковского. Пространство маленькое, 10 на 2 на 2, поэтому Optuna и полный перебор здесь должны сойтись к одному и тому же результату

In [7]:
from sklearn.neighbors import KNeighborsRegressor


def build_knn(trial: optuna.Trial) -> Pipeline:
    """Собирает пайплайн KNN с параметрами, подобранными Optuna"""
    n_neighbors = trial.suggest_int("n_neighbors", 3, 51)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    p = trial.suggest_categorical("p", [1, 2])
    return make_pipeline(build_preprocessor(scale=True), KNeighborsRegressor(n_neighbors=n_neighbors, weights=weights, p=p))


knn_study = optuna_tune_and_evaluate("4. KNN", build_knn, n_trials=40)

4. KNN: RMSE 0.1657, std 0.0082, лучшие параметры {'n_neighbors': 8, 'weights': 'distance', 'p': 1}


### Выводы: KNN

- RMSE 0.1657, чуть лучше, чем 0.1661 у полного перебора в прошлом ноутбуке
- Лучшие параметры почти совпадают: 8 соседей вместо 7, тот же вес по расстоянию и та же манхэттенская метрика
- Как и ожидалось для маленького пространства, 40 сочетаний в сетке против 40 попыток Optuna дают практически одинаковый результат

## 5. Дерево решений

В прошлом ноутбуке гиперпараметры перебирались по тем же спискам значений, что и в `RandomizedSearchCV`, для честного сравнения на маленьких пространствах, у Ridge, Lasso, ElasticNet и KNN это было оправдано. Для дерева так делать не стоит: Optuna умеет искать внутри непрерывного диапазона и сама сосредотачивается на удачных областях по ходу поиска, а если дать ей тот же дискретный список, что и раньше, она в лучшем случае найдёт то же самое, а не точку между старыми значениями или за пределами прежнего диапазона

Поэтому числовые параметры здесь заданы диапазоном, а не списком: `max_depth` от 2 до 30, `min_samples_split` от 2 до 50, `min_samples_leaf` от 1 до 30, `max_features` от 0.03 до 1.0 в логарифмической шкале, чтобы охватить и малые доли вроде `sqrt`/`log2` от 230 колонок, и все признаки сразу, `ccp_alpha` от 0 до 0.02. `criterion` и `splitter` остаются списком, потому что это категории, а не числа

In [8]:
from sklearn.tree import DecisionTreeRegressor


def build_tree(trial: optuna.Trial) -> Pipeline:
    """Собирает пайплайн дерева решений с параметрами, подобранными Optuna"""
    model = DecisionTreeRegressor(
        criterion=trial.suggest_categorical("criterion", ["squared_error", "absolute_error", "poisson"]),
        splitter=trial.suggest_categorical("splitter", ["best", "random"]),
        max_depth=trial.suggest_int("max_depth", 2, 30),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 50),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 30),
        max_features=trial.suggest_float("max_features", 0.03, 1.0, log=True),
        ccp_alpha=trial.suggest_float("ccp_alpha", 0.0, 0.02),
        random_state=RANDOM_STATE,
    )
    return make_pipeline(build_preprocessor(scale=False), model)


tree_study = optuna_tune_and_evaluate("5. Дерево решений", build_tree, n_trials=150)

5. Дерево решений: RMSE 0.1957, std 0.0124, лучшие параметры {'criterion': 'squared_error', 'splitter': 'random', 'max_depth': 21, 'min_samples_split': 6, 'min_samples_leaf': 11, 'max_features': 0.7400797802661504, 'ccp_alpha': 9.652527872525653e-05}


### Выводы: дерево решений

- RMSE 0.1957, std 0.0124. Хуже, чем 0.1853 у полного перебора в прошлом ноутбуке, и разброс между фолдами заметно больше, 0.0124 против 0.0078
- Лучшие параметры: `splitter` найден `random`, а не `best`, который выигрывал в прошлом ноутбуке
- Похоже на слабое место Optuna именно для этой задачи, а не ошибку в коде. Одно дерево без ансамбля само по себе сильно скачет от небольших изменений параметров, а `TPESampler` строит вероятностную модель "удачных" и "неудачных" попыток по категориям вроде `splitter` так же, как по числам. Если несколько первых попыток со `splitter=random` случайно оказались удачными, дальнейший поиск мог начать предлагать `random` чаще, чем стоило, и не выправиться за 150 попыток. Полный перебор в прошлом ноутбуке видел `best` и `random` поровну по построению, поэтому не попал в эту ловушку
- Вывод не в пользу узкого сужения диапазонов заранее: раз даже сравнительно широкий поиск с непрерывными диапазонами оступился на выборе категории, заранее сузить диапазон вокруг найденного в переборе значения было бы ещё рискованнее
- Для одного дерева результат `RandomizedSearchCV` из прошлого ноутбука, 0.1853, надёжнее, чем найденный здесь. Это не значит, что Optuna хуже как метод вообще, скорее что для моделей с высокой дисперсией вроде одного дерева ей нужно больше попыток или явный контроль за тем, чтобы категориальные варианты пробовались не только там, где ранние попытки оказались удачными

## 6. Случайный лес

Диапазоны берём непрерывными, как для дерева, и по тем же причинам, что и в прошлом ноутбуке, исключаем часть вариантов ради практичности:

- `criterion` только `squared_error` и `poisson`, `absolute_error` слишком медленный, будучи умноженным на сотни деревьев
- `max_features` до 0.9, а не до 1.0: полный набор признаков на каждом разбиении был тем самым медленным и нестабильным случаем в прошлом ноутбуке, здесь эта область осознанно исключена, а не просто не найдена
- `bootstrap` фиксирован как `True`, это стандартный механизм случайного леса, а не отдельный гиперпараметр

У `criterion` только две категории, и после находки с деревом решений увеличиваем `n_startup_trials` до 25 из 60 попыток, чтобы обе категории точно успели пройти через случайную часть поиска до того, как Optuna начнёт опираться на находки

Здесь, в отличие от прошлого ноутбука, у `RandomForestRegressor` можно спокойно ставить `n_jobs=-1`: Optuna перебирает попытки одну за другой, без внешнего `joblib`, поэтому нет того конфликта вложенной многопроцессности, из-за которого раньше падал воркер

In [9]:
from sklearn.ensemble import RandomForestRegressor


def build_random_forest(trial: optuna.Trial) -> Pipeline:
    """Собирает пайплайн случайного леса с параметрами, подобранными Optuna"""
    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        criterion=trial.suggest_categorical("criterion", ["squared_error", "poisson"]),
        max_depth=trial.suggest_int("max_depth", 3, 25),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 50),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 30),
        max_features=trial.suggest_float("max_features", 0.05, 0.9, log=True),
        max_samples=trial.suggest_float("max_samples", 0.3, 1.0),
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return make_pipeline(build_preprocessor(scale=False), model)


random_forest_study = optuna_tune_and_evaluate(
    "6. Случайный лес", build_random_forest, n_trials=60, n_startup_trials=25
)

6. Случайный лес: RMSE 0.1318, std 0.0046, лучшие параметры {'n_estimators': 118, 'criterion': 'poisson', 'max_depth': 23, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.45997403291690353, 'max_samples': 0.9111748662471574}


### Выводы: случайный лес

- RMSE 0.1318, std 0.0046. Лучше, чем 0.1376 в прошлом ноутбуке, и разброс между фолдами меньше, 0.0046 против 0.0050
- Лучшие параметры: глубина 23, минимум 1 дом в листе, `criterion` `poisson`, но всего 118 деревьев, `max_features` 0.46, `max_samples` 0.91
- Эта находка не укладывается в прежний вывод про то, что неглубокие деревья выигрывают. Здесь наоборот, глубина почти не ограничена, а листья могут состоять из одного дома. Вероятная причина в том, что для случайного леса регуляризация идёт не только через глубину дерева, но и через `max_features` и `max_samples`: если у каждого дерева и так ограничен набор признаков и часть домов через сэмплирование, само дерево может расти свободно, ансамбль всё равно останется разнообразным
- И `n_startup_trials`, поднятый до 25 из 60 после находки с деревом решений, здесь сработал: `criterion` `poisson` выигрывает не потому, что ему повезло с первыми попытками, у обеих категорий было много случайных проверок до того, как поиск начал делать выводы
- В отличие от одного дерева результату здесь можно доверять больше: ансамбль сам по себе устойчивее к шуму отдельных попыток поиска, что и подтверждает меньший разброс между фолдами

## 7. LightGBM

Тот же набор параметров, что в прошлом ноутбуке, но диапазонами, а не списками. `max_depth` без варианта "без ограничения": для LightGBM основной рычаг сложности дерева и так `num_leaves`, а не глубина, `max_depth` здесь больше подстраховка сверху

`boosting_type` снова всего две категории, `gbdt` и `dart`, поднимаем `n_startup_trials` до 25 из 70 попыток по той же причине, что и для леса

In [10]:
from lightgbm import LGBMRegressor


def build_lightgbm(trial: optuna.Trial) -> Pipeline:
    """Собирает пайплайн LightGBM с параметрами, подобранными Optuna"""
    model = LGBMRegressor(
        boosting_type=trial.suggest_categorical("boosting_type", ["gbdt", "dart"]),
        num_leaves=trial.suggest_int("num_leaves", 4, 200, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 15),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        n_estimators=trial.suggest_int("n_estimators", 50, 1500, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 2, 60),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        subsample_freq=trial.suggest_int("subsample_freq", 0, 7),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 5, log=True),
        min_split_gain=trial.suggest_float("min_split_gain", 1e-8, 0.1, log=True),
        random_state=RANDOM_STATE,
        verbose=-1,
    )
    return make_pipeline(build_preprocessor(scale=False), model)


lightgbm_study = optuna_tune_and_evaluate("7. LightGBM", build_lightgbm, n_trials=70, n_startup_trials=25)

7. LightGBM: RMSE 0.1150, std 0.0061, лучшие параметры {'boosting_type': 'gbdt', 'num_leaves': 135, 'max_depth': 4, 'learning_rate': 0.014191639087470571, 'n_estimators': 1310, 'min_child_samples': 4, 'subsample': 0.8155726506796872, 'subsample_freq': 5, 'colsample_bytree': 0.43429044707904246, 'reg_alpha': 0.0005811844267242152, 'reg_lambda': 0.013591235587074845, 'min_split_gain': 0.0012512344321506698}


### Выводы: LightGBM

- RMSE 0.1150, std 0.0061. Заметно лучше, чем 0.1199 в прошлом ноутбуке, и это лучший результат среди всех деревьев и бустингов на сегодня, лучше даже CatBoost (0.1165) из прошлого ноутбука
- Лучшие параметры: `learning_rate` маленький, 0.014, а `n_estimators` большой, 1310, `max_depth` всего 4, `num_leaves` 135
- Это прямое подтверждение догадки из прошлого ноутбука: там `learning_rate` и `n_estimators` выбирались независимо друг от друга в случайном переборе, и часть из 100 сочетаний тратилась на заведомо неудачные пары вроде большого шага с большим числом деревьев. Здесь Optuna, оценивая результаты предыдущих попыток, сама нашла согласованную пару: маленький шаг компенсируется большим числом деревьев, и именно это, а не какой-то один параметр по отдельности, дало основной прирост
- Неглубокие деревья снова выигрывают, `max_depth` 4, это уже третье подтверждение подряд после XGBoost и CatBoost в прошлом ноутбуке. Отдельный случай леса, где выигрывали глубокие деревья, остаётся исключением, а не опровержением закономерности
- LightGBM пока лучшая модель из ансамблей деревьев в обоих ноутбуках, но всё ещё не обгоняет Ridge (0.1117) и Lasso (0.1111)

## 8. XGBoost

Как и для LightGBM, переводим сетку прошлого ноутбука в непрерывные диапазоны и добавляем `booster` как категориальный параметр, поэтому поднимаем `n_startup_trials` заранее, а не по факту неудачи, как это было с деревом решений

In [11]:
from xgboost import XGBRegressor


def build_xgboost(trial: optuna.Trial) -> Pipeline:
    model = XGBRegressor(
        booster=trial.suggest_categorical("booster", ["gbtree", "dart"]),
        n_estimators=trial.suggest_int("n_estimators", 50, 1500, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        min_child_weight=trial.suggest_float("min_child_weight", 0.5, 20, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        gamma=trial.suggest_float("gamma", 1e-8, 2.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 5, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 5, log=True),
        random_state=RANDOM_STATE,
        verbosity=0,
    )
    return make_pipeline(build_preprocessor(scale=False), model)


xgboost_study = optuna_tune_and_evaluate("8. XGBoost", build_xgboost, n_trials=70, n_startup_trials=25)

8. XGBoost: RMSE 0.1159, std 0.0059, лучшие параметры {'booster': 'gbtree', 'n_estimators': 990, 'max_depth': 3, 'learning_rate': 0.05847892915316512, 'min_child_weight': 0.637689473011825, 'subsample': 0.9029473470866576, 'colsample_bytree': 0.4607459901880062, 'gamma': 8.285973574448373e-08, 'reg_alpha': 4.032976287219069e-07, 'reg_lambda': 3.6499272926641786}


### Выводы: XGBoost

- RMSE 0.1159, std 0.0059. Лучше, чем 0.1169 в прошлом ноутбуке, и лучше CatBoost оттуда же (0.1165), но немного хуже LightGBM здесь (0.1150)
- Лучшие параметры: booster остался gbtree, а не dart, при 25 стартовых попытках на два варианта категориального параметра это не случайное везение, а честный выбор
- max_depth снова маленький, 3, ещё одно подтверждение того, что на этих данных неглубокие деревья выигрывают у глубоких, теперь это видно уже в трёх бустингах и двух ноутбуках подряд
- gamma и reg_alpha ушли почти к нулю, а основным сдерживающим фактором стал reg_lambda, то есть регуляризация здесь работает через L2, а не через порог на выигрыш от разбиения или L1
- Порядок пока не меняется: линейные модели Ridge и Lasso впереди, LightGBM лучший из бустингов, XGBoost и CatBoost почти вровень следом

## Итоги ноутбука

CatBoost решили пропустить: даже при `thread_count=2` один фит стоит заметно дороже, чем у LightGBM или XGBoost, а Optuna из-за условного `bootstrap_type` требует ещё и повышенного `n_startup_trials`, вместе это уже не укладывается в разумное время ожидания. Результат CatBoost из прошлого ноутбука (0.1165, `RandomizedSearchCV`) остаётся в силе как единственная оценка для этой модели

Все результаты Optuna рядом с результатами `GridSearchCV`/`RandomizedSearchCV` из `classical_models.ipynb`, от лучшей модели к худшей по Optuna:

| Модель | Optuna RMSE | Grid/RandomizedSearchCV RMSE |
| --- | --- | --- |
| Lasso | 0.1110 | 0.1111 |
| ElasticNet | 0.1111 | 0.1111 |
| Ridge | 0.1117 | 0.1117 |
| LightGBM | 0.1150 | 0.1199 |
| XGBoost | 0.1159 | 0.1169 |
| CatBoost | — (пропущено) | 0.1165 |
| Случайный лес | 0.1318 | 0.1376 |
| KNN | 0.1657 | 0.1661 |
| Дерево решений | 0.1957 | 0.1853 |

Что видно при сравнении двух подходов к поиску гиперпараметров:

- **На линейных моделях разницы почти нет.** У них всего один-два гиперпараметра на гладкой шкале, там любой достаточно плотный перебор находит один и тот же оптимум, Grid и Optuna сходятся к практически одному числу
- **На бустингах Optuna выигрывает заметно.** LightGBM улучшился с 0.1199 до 0.1150, XGBoost с 0.1169 до 0.1159. Причина не в самом алгоритме поиска, а в том, что непрерывные диапазоны позволяют найти согласованную пару `learning_rate`/`n_estimators`, тогда как отдельные дискретные списки этих двух параметров в `RandomizedSearchCV` часть попыток тратят на заведомо неудачные комбинации
- **Единственное ухудшение — дерево решений**, и оно не про Optuna как метод, а про то, что для одного категориального параметра со значимым перекосом (`splitter`) нужно намного больше случайных стартовых попыток, чем задано по умолчанию. С поднятым `n_startup_trials` эта проблема не возникла ни в случайном лесе, ни в LightGBM, ни в XGBoost
- **Порядок моделей не изменился.** Ridge и Lasso остаются лучшими моделями в обоих ноутбуках, LightGBM теперь ближе всех к ним среди деревьев и бустингов, а само дерево решений — по-прежнему худшая модель

Общий вывод: непрерывный поиск полезен именно там, где у модели много гиперпараметров с совместным эффектом, как `learning_rate` и `n_estimators` у бустингов, а для моделей с одним-двумя гиперпараметрами хватает и обычной сетки